## 1. Andmestiku laadimine ja esmane ülevaade

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv("takso_andmed.csv") ## andmestiku sisselugemine

In [6]:
df.head() ## esimesed 5 rida    

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_token,rider_app_version,order_state,order_try_state,driver_app_version,driver_device_uid_new,device_name,eu_indicator,overpaid_ride_ticket,fraud_score
0,22,22,2020-02-02 3:37:31,4.04,10.0,2839,700,1,client,finished,...,NaN,CI.4.17,finished,finished,DA.4.37,1596,Xiaomi Redmi 6,1,0,-1383.0
1,618,618,2020-02-08 2:26:19,6.09,3.6,5698,493,1,client,finished,...,NaN,CA.5.43,finished,finished,DA.4.39,1578,Samsung SM-G965F,1,0,NaN
2,657,657,2020-02-08 11:50:35,4.32,3.5,4426,695,1,client,finished,...,NaN,CA.5.43,finished,finished,DA.4.37,951,Samsung SM-A530F,1,0,-166.0
3,313,313,2020-02-05 6:34:54,72871.72,NaN,49748,1400,0,client,finished,...,NaN,CA.5.23,finished,finished,DA.4.37,1587,TECNO-Y6,0,1,NaN
4,1176,1176,2020-02-13 17:31:24,20032.50,19500.0,10273,5067,1,client,finished,...,NaN,CA.5.04,finished,finished,DA.4.37,433,Itel W5504,0,0,NaN


In [7]:
df_clean = df.copy() ## teeme koopia, töötame edasi df_cleaniga

### Andmestiku suurus, veerud ja andmetüübid

In [8]:
df_clean.info() ## andmestiku info

<class 'pandas.DataFrame'>
RangeIndex: 4943 entries, 0 to 4942
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id_new           4943 non-null   int64  
 1   order_try_id_new       4943 non-null   int64  
 2   calc_created           4943 non-null   str    
 3   metered_price          4923 non-null   float64
 4   upfront_price          3409 non-null   float64
 5   distance               4943 non-null   int64  
 6   duration               4943 non-null   int64  
 7   gps_confidence         4943 non-null   int64  
 8   entered_by             4943 non-null   str    
 9   b_state                4943 non-null   str    
 10  dest_change_number     4943 non-null   int64  
 11  prediction_price_type  4923 non-null   str    
 12  predicted_distance     4923 non-null   float64
 13  predicted_duration     4923 non-null   float64
 14  change_reason_pricing  298 non-null    str    
 15  ticket_id_new  

## 2. Andmete esmane korrastamine

### Kuupäeva ja kellaaja korrastamine


In [9]:
df_clean["calc_created"] = pd.to_datetime(df_clean["calc_created"]) ## muudame kuupäeva õigeks tüübiks

In [10]:
df_clean["calc_created"].dtype ## seejärel kontrollin, kas on õige tüüp nüüd

dtype('<M8[us]')

In [11]:
# Eraldan kuupäevast Power BI analüüsi jaoks
# kuupäeva, kellaaja, nädalapäeva, kuu ja aasta.

df_clean["date"] = df_clean["calc_created"].dt.date
df_clean["time"] = df_clean["calc_created"].dt.time

df_clean["day_of_week"] = df_clean["calc_created"].dt.day_name()
df_clean["day_of_week_nr"] = df_clean["calc_created"].dt.dayofweek + 1

df_clean["month"] = df_clean["calc_created"].dt.month
df_clean["year"] = df_clean["calc_created"].dt.year

In [12]:
df_clean[
    ["calc_created", "date", "time", "day_of_week",
     "day_of_week_nr", "month", "year"]
].head() 

## kontrollime kas eraldas kuupäeva

,calc_created,date,time,day_of_week,day_of_week_nr,month,year
0,2020-02-02 03:37:31,2020-02-02,03:37:31,Sunday,7,2,2020
1,2020-02-08 02:26:19,2020-02-08,02:26:19,Saturday,6,2,2020
2,2020-02-08 11:50:35,2020-02-08,11:50:35,Saturday,6,2,2020
3,2020-02-05 06:34:54,2020-02-05,06:34:54,Wednesday,3,2,2020
4,2020-02-13 17:31:24,2020-02-13,17:31:24,Thursday,4,2,2020


In [13]:
# Kontrollin, kas Power BI jaoks loodud ajaveerud on kõik olemas
time_columns = ["date", "time", "day_of_week", "day_of_week_nr", "month", "year"]

[col for col in time_columns if col in df_clean.columns]

['date', 'time', 'day_of_week', 'day_of_week_nr', 'month', 'year']

### Ebavajalike veergude eemaldamine


In [14]:
df_clean = df_clean.drop(columns=["device_token"])  ## eemaldan täiesti tühja veeru

In [15]:
df_clean.shape ## vaatame kui suur table nüüd on 

(4943, 31)

## 3. Andmekvaliteedi kontroll

### Puuduvate väärtuste ülevaade

In [16]:
# Kontrollin, millistes veergudes esineb puuduvaid väärtusi
# ning kui suure osa andmetest need moodustavad.

missing_values = pd.DataFrame({
    "Puuduvaid": df_clean.isna().sum(),
    "Osakaal (%)": (df_clean.isna().mean() * 100).round(2)
})

missing_values[
    missing_values["Puuduvaid"] > 0
].sort_values("Puuduvaid", ascending=False)

,Puuduvaid,Osakaal (%)
change_reason_pricing,4645,93.97
fraud_score,2759,55.82
upfront_price,1534,31.03
metered_price,20,0.40
predicted_distance,20,0.40
prediction_price_type,20,0.40
predicted_duration,20,0.40
rider_app_version,16,0.32


### Identsete ridade kontroll

In [17]:
df_clean.duplicated().sum()  ## kontrollin, mitu täielikult identset rida on

np.int64(0)

### Identsete veergude kontroll


In [18]:
# Kontrollin, kas andmestikus on sama nimega veerge

df_clean.columns.duplicated().sum()

np.int64(0)

In [19]:
# Kuvan kõik veerunimed, et kontrollida nende nimetusi ja võimalikke ebakõlasid

df_clean.columns.tolist()

['order_id_new',
 'order_try_id_new',
 'calc_created',
 'metered_price',
 'upfront_price',
 'distance',
 'duration',
 'gps_confidence',
 'entered_by',
 'b_state',
 'dest_change_number',
 'prediction_price_type',
 'predicted_distance',
 'predicted_duration',
 'change_reason_pricing',
 'ticket_id_new',
 'rider_app_version',
 'order_state',
 'order_try_state',
 'driver_app_version',
 'driver_device_uid_new',
 'device_name',
 'eu_indicator',
 'overpaid_ride_ticket',
 'fraud_score',
 'date',
 'time',
 'day_of_week',
 'day_of_week_nr',
 'month',
 'year']

In [20]:
# Kontrollin, kas erineva nimega veergudes on täpselt sama sisu
# Kui leitakse identsed veerud, kuvab Python nende nimed 

for i, col1 in enumerate(df_clean.columns):
    for col2 in df_clean.columns[i + 1:]:
        if df_clean[col1].equals(df_clean[col2]):
            print(col1, "=", col2)

b_state = order_try_state


**Järeldus:** Andmestikus leidub identse sisuga, kuid erineva tähendusega veerge. 
Neid ei eemaldatud automaatselt, sest veergude äriline tähendus on erinev.

### Korduvate tellimuste kontroll

In [21]:
# Kontrollin, kui paljudel ridadel erinevad tellimuse ID ja sõidukatse ID.
# See aitab mõista, kas üks tellimus võib sisaldada mitut eraldi katset.

(df_clean["order_id_new"] != df_clean["order_try_id_new"]).sum()

np.int64(58)

In [22]:
# Kuvan mõned read, kus order_id ja order_try_id erinevad,
# et näha, millistel juhtudel see esineb.

df_clean[
    df_clean["order_id_new"] != df_clean["order_try_id_new"]
].head(10)

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_name,eu_indicator,overpaid_ride_ticket,fraud_score,date,time,day_of_week,day_of_week_nr,month,year
52,1608,1609,2020-02-17 08:45:21,4087.50,4500.0,4752,1755,1,client,finished,...,TECNO WX3,0,0,NaN,2020-02-17,08:45:21,Monday,1,2,2020
67,536,537,2020-02-07 15:12:43,11323.02,8500.0,8573,1168,0,client,finished,...,TECNO MOBILE LIMITED TECNO B1,0,1,NaN,2020-02-07,15:12:43,Friday,5,2,2020
93,1716,1715,2020-02-18 13:07:14,2.54,3.3,2193,352,1,client,finished,...,Samsung SM-G532F,1,0,-246.0,2020-02-18,13:07:14,Tuesday,2,2,2020
318,2024,2023,2020-02-21 15:48:59,16337.46,NaN,8688,3827,1,client,finished,...,Samsung SAMSUNG-SM-G900A,0,1,NaN,2020-02-21,15:48:59,Friday,5,2,2020
332,1255,1254,2020-02-14 13:40:31,2.92,3.0,2407,366,1,client,finished,...,Samsung SM-J700H,1,0,-90.0,2020-02-14,13:40:31,Friday,5,2,2020
338,150,149,2020-02-03 12:17:59,20726.77,25000.0,14339,2420,0,client,finished,...,TECNO MOBILE LIMITED TECNO B1,0,0,NaN,2020-02-03,12:17:59,Monday,1,2,2020
447,2024,2023,2020-02-21 15:48:59,16337.46,NaN,8688,3827,1,client,finished,...,Samsung SAMSUNG-SM-G900A,0,0,NaN,2020-02-21,15:48:59,Friday,5,2,2020
467,2332,2331,2020-02-23 18:04:24,13.11,12.9,20945,1451,1,client,finished,...,Google Pixel 2 XL,1,0,-1102.0,2020-02-23,18:04:24,Sunday,7,2,2020
628,1254,1255,2020-02-14 14:28:34,14533.50,8000.0,8118,3123,0,client,finished,...,Xiaomi Redmi Note 7 Pro,0,0,NaN,2020-02-14,14:28:34,Friday,5,2,2020
638,402,403,2020-02-06 06:22:57,7894.13,9000.0,4266,1000,1,client,finished,...,TECNO MOBILE LIMITED TECNO LC6,0,0,NaN,2020-02-06,06:22:57,Thursday,4,2,2020


In [23]:
df_clean["order_id_new"].duplicated().sum() ## kontrollin kui palju kordub tellimusi

np.int64(777)

In [24]:
# Loendan kõigepealt, mitu korda iga order_id_new andmestikus esineb.
order_id_counts = df_clean["order_id_new"].value_counts()

# Seejärel loendan, mitu order ID-d esineb 1x, 2x, 3x jne.
order_id_frequency = order_id_counts.value_counts().sort_index()

# Kuvan tulemuse.
order_id_frequency

count
1    3554
2     487
3      94
4      24
5       5
6       2
Name: count, dtype: int64

#### Korduvate order_id väärtuste uurimine

In [25]:
## kuva kõik korduvate order ID-de read

duplicate_orders = df_clean[
    df_clean["order_id_new"].duplicated(keep=False)
]

duplicate_orders.sort_values("order_id_new").head(20)

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_name,eu_indicator,overpaid_ride_ticket,fraud_score,date,time,day_of_week,day_of_week_nr,month,year
1532,3,3,2020-02-02 00:49:24,14.87,NaN,15541,1690,0,client,finished,...,Samsung SM-J415FN,1,0,-1516.0,2020-02-02,00:49:24,Sunday,7,2,2020
1902,3,3,2020-02-02 00:49:24,14.87,NaN,15541,1690,0,client,finished,...,Samsung SM-J415FN,1,0,-1516.0,2020-02-02,00:49:24,Sunday,7,2,2020
2299,13,13,2020-02-02 02:31:56,8.89,NaN,16880,1339,1,driver,finished,...,HUAWEI P7-L10,1,0,-196.0,2020-02-02,02:31:56,Sunday,7,2,2020
966,13,13,2020-02-02 02:31:56,8.89,NaN,16880,1339,1,driver,finished,...,HUAWEI P7-L10,1,0,-196.0,2020-02-02,02:31:56,Sunday,7,2,2020
4636,19,19,2020-02-02 02:59:48,6.69,5.3,9498,793,1,client,finished,...,Xiaomi Redmi Note 7,1,0,-316.0,2020-02-02,02:59:48,Sunday,7,2,2020
4160,19,19,2020-02-02 02:59:48,6.69,5.3,9498,793,1,client,finished,...,Xiaomi Redmi Note 7,1,0,-316.0,2020-02-02,02:59:48,Sunday,7,2,2020
4861,26,26,2020-02-02 06:44:20,6000.00,8500.0,12,160,1,client,finished,...,TECNO MOBILE LIMITED TECNO KC8,0,0,NaN,2020-02-02,06:44:20,Sunday,7,2,2020
2549,26,26,2020-02-02 06:44:20,6000.00,8500.0,12,160,1,client,finished,...,TECNO MOBILE LIMITED TECNO KC8,0,0,NaN,2020-02-02,06:44:20,Sunday,7,2,2020
2585,26,26,2020-02-02 06:44:20,6000.00,8500.0,12,160,1,client,finished,...,TECNO MOBILE LIMITED TECNO KC8,0,0,NaN,2020-02-02,06:44:20,Sunday,7,2,2020
362,29,29,2020-02-02 05:57:43,7940.22,7500.0,4869,1122,1,client,finished,...,Samsung SM-N920T,0,0,NaN,2020-02-02,05:57:43,Sunday,7,2,2020


In [26]:
df_clean[df_clean["order_id_new"] == 3].T 
## .T keerab tabeli lihtsalt teistpidi, et kõiki 30+ veergu oleks lihtne võrrelda. 
## Vaatasin, miks ticket_id_new kordub
## Leidin et customer support ticket ID on siiski erinev, seega võib olla seotud mitu erinevat klienditoe ticket'it.

,1532,1902
order_id_new,3,3
order_try_id_new,3,3
calc_created,2020-02-02 00:49:24,2020-02-02 00:49:24
metered_price,14.87,14.87
upfront_price,NaN,NaN
distance,15541,15541
duration,1690,1690
gps_confidence,0,0
entered_by,client,client
b_state,finished,finished


Leidsin, et vähemalt osadel korduvatel order_id väärtustel on erinevad ticket_id väärtused, mis viitab sellele, et ühe tellimusega võib olla seotud mitu tugipiletit. Seetõttu ei käsitle ma korduvaid order_id väärtusi automaatselt duplikaatridadena.

## 4. Kategooriliste väärtuste kontroll

### Sõidu staatuste kontroll


In [27]:
# Kontrollin, millised sõidu olekud (b_state) andmetes esinevad

df_clean["b_state"].value_counts(dropna=False)

b_state
finished    4943
Name: count, dtype: int64

### Sihtkoha sisestaja kontroll

In [28]:
# Kontrollin, kes sisestas sihtkoha aadressi ja kui palju iga väärtust esineb

df_clean["entered_by"].value_counts(dropna=False)

entered_by
client      4722
driver       216
reseller       5
Name: count, dtype: int64

### Hinnaprognoosi tüüpide kontroll

In [29]:
# Kontrollin, millised hinnaprognoosi tüübid andmetes esinevad
# dropna=False näitab tulemuses ka puuduvaid väärtusi (NaN)

df_clean["prediction_price_type"].value_counts(dropna=False)

prediction_price_type
upfront                        3432
prediction                     1279
upfront_destination_changed     208
NaN                              20
upfront_waypoint_changed          4
Name: count, dtype: int64

### Tellimuse staatuste kontroll

In [30]:
# Kontrollin tellimuste staatuste väärtusi

df_clean["order_state"].value_counts(dropna=False)

order_state
finished    4942
active         1
Name: count, dtype: int64

#### Aktiivse tellimuse uurimine

In [31]:
# Vaatan üle ainsa tellimuse, mille order_state ei ole "finished"
df_clean[df_clean["order_state"] == "active"].T

,1296
order_id_new,457
order_try_id_new,457
calc_created,2020-02-06 18:30:35
metered_price,22071.02
upfront_price,12500.0
distance,16986
duration,1547
gps_confidence,0
entered_by,client
b_state,finished


### Order try staatuste kontroll

In [32]:
# Kontrollin order try staatuste väärtusi

df_clean["order_try_state"].value_counts(dropna=False)

order_try_state
finished    4943
Name: count, dtype: int64

## 5. Numbriliste väärtuste esmane kontroll

### Numbriliste tunnuste kirjeldav statistika

In [33]:
# Vaatan oluliste numbriliste veergude põhilist statistikat,
# et leida võimalikke ebaloogilisi või äärmuslikke väärtusi

df_clean[
    [
        "metered_price",
        "upfront_price",
        "distance",
        "duration",
        "predicted_distance",
        "predicted_duration",
        "fraud_score"
    ]
].describe()

,metered_price,upfront_price,distance,duration,predicted_distance,predicted_duration,fraud_score
count,4923.000000,3409.000000,4943.000000,4943.000000,4923.000000,4923.000000,2184.000000
mean,7998.471296,4160.095747,9769.223144,1566.230629,8822.636807,1106.737355,-674.046703
std,15815.850352,17015.711912,10912.426401,1650.329858,10548.801733,806.098535,1119.189890
min,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,-14225.000000
25%,5.380000,4.200000,3785.500000,604.000000,4130.500000,597.500000,-826.500000
50%,13.350000,6.600000,7140.000000,1054.000000,6918.000000,939.000000,-278.500000
75%,10991.670000,4000.000000,11953.000000,1929.500000,10674.000000,1427.000000,-64.750000
max,194483.520000,595000.000000,233190.000000,22402.000000,353538.000000,20992.000000,49.000000


### Negatiivsete väärtuste kontroll

In [34]:
# Kontrollin, kas hinnas, distantsis või kestuses esineb
# negatiivseid väärtusi, mis võivad viidata vigastele andmetele

numeric_columns = [
    "metered_price",
    "upfront_price",
    "distance",
    "duration",
    "predicted_distance",
    "predicted_duration"
]

for column in numeric_columns:
    print(column, "negative values:", (df_clean[column] < 0).sum())

metered_price negative values: 0
upfront_price negative values: 0
distance negative values: 0
duration negative values: 0
predicted_distance negative values: 0
predicted_duration negative values: 0


## 6. Puuduvate hindade analüüs

### Puuduva 'metered_price' analüüs

In [35]:
# Kuvan kõik read, kus metered_price puudub,
# koos KÕIGI andmestiku veergudega, et saaksin uurida,
# mis neid 20 kirjet teistest eristab

missing_metered = df_clean[df_clean["metered_price"].isna()]

missing_metered

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_name,eu_indicator,overpaid_ride_ticket,fraud_score,date,time,day_of_week,day_of_week_nr,month,year
64,217,217,2020-02-04 08:07:10,NaN,NaN,6249,2477,1,driver,finished,...,Xiaomi Redmi 8,1,0,NaN,2020-02-04,08:07:10,Tuesday,2,2,2020
393,3066,3066,2020-03-02 17:45:17,NaN,NaN,5483,917,1,driver,finished,...,HUAWEI EML-L29,1,0,NaN,2020-03-02,17:45:17,Monday,1,3,2020
458,3320,3320,2020-03-06 03:33:52,NaN,NaN,3364,323,0,reseller,finished,...,HUAWEI CLT-L29,1,0,NaN,2020-03-06,03:33:52,Friday,5,3,2020
513,1166,1166,2020-02-13 14:55:14,NaN,NaN,21997,1742,1,driver,finished,...,HUAWEI SLA-L22,1,0,-74.0,2020-02-13,14:55:14,Thursday,4,2,2020
779,1759,1759,2020-02-19 00:55:07,NaN,NaN,10309,849,1,reseller,finished,...,"iPhone8,1",1,0,NaN,2020-02-19,00:55:07,Wednesday,3,2,2020
998,861,861,2020-02-10 06:52:29,NaN,NaN,1424,416,1,driver,finished,...,Samsung SM-G973F,1,0,NaN,2020-02-10,06:52:29,Monday,1,2,2020
1206,1349,1349,2020-02-14 23:08:22,NaN,NaN,11206,1268,1,client,finished,...,Samsung SM-T295,1,0,-1270.0,2020-02-14,23:08:22,Friday,5,2,2020
1287,2012,2012,2020-02-21 13:24:40,NaN,NaN,3717,713,1,driver,finished,...,Samsung SM-A530F,1,0,NaN,2020-02-21,13:24:40,Friday,5,2,2020
1340,1112,1112,2020-02-12 23:32:11,NaN,NaN,4114,528,1,driver,finished,...,Xiaomi Mi A1,1,0,NaN,2020-02-12,23:32:11,Wednesday,3,2,2020
1582,460,460,2020-02-06 19:34:43,NaN,NaN,6718,764,1,client,finished,...,"iPhone8,1",1,0,NaN,2020-02-06,19:34:43,Thursday,4,2,2020


#### Puuduva 'metered_price' seoste uurimine

In [36]:
# Uurin 20 rida, kus metered_price puudub.
# Vaatan, kas neil on ühiseid tunnuseid, mis võiksid selgitada puuduvat hinda.

missing_metered[
    [
        "b_state",
        "order_state",
        "order_try_state",
        "prediction_price_type",
        "entered_by",
        "gps_confidence",
        "eu_indicator"
    ]
].value_counts(dropna=False)

b_state   order_state  order_try_state  prediction_price_type  entered_by  gps_confidence  eu_indicator
finished  finished     finished         NaN                    driver      1               1               13
                                                               reseller    1               1                3
                                                                           0               1                2
                                                               client      1               1                2
Name: count, dtype: int64

In [37]:
# Kontrollin, millised olulised väärtused puuduvad
# nendel samadel 20 real

missing_metered.isnull().sum().sort_values(ascending=False)

metered_price            20
upfront_price            20
change_reason_pricing    20
prediction_price_type    20
predicted_distance       20
predicted_duration       20
fraud_score              18
rider_app_version        16
distance                  0
duration                  0
calc_created              0
dest_change_number        0
b_state                   0
order_try_id_new          0
order_id_new              0
gps_confidence            0
entered_by                0
order_state               0
ticket_id_new             0
driver_app_version        0
driver_device_uid_new     0
device_name               0
order_try_state           0
eu_indicator              0
overpaid_ride_ticket      0
date                      0
time                      0
day_of_week               0
day_of_week_nr            0
month                     0
year                      0
dtype: int64

In [38]:
# Vaatan 20 erandliku sõidu device_name väärtusi,
# et kontrollida, kas puuduvad andmed võivad olla seotud
# konkreetse seadme või platvormiga

missing_metered["device_name"].value_counts(dropna=False)

device_name
iPhone8,1            2
Samsung SM-G973F     2
Sony G8441           2
Xiaomi Redmi 8       1
HUAWEI EML-L29       1
HUAWEI CLT-L29       1
HUAWEI SLA-L22       1
Samsung SM-T295      1
Samsung SM-A530F     1
Xiaomi Mi A1         1
Xiaomi Redmi 6A      1
Samsung SM-A705FN    1
CUBOT_P20            1
Samsung SM-G935F     1
HUAWEI MAR-LX1A      1
Samsung SM-J510FN    1
Samsung SM-A405FN    1
Name: count, dtype: int64

In [39]:
# Vaatan nende 20 sõidu rider_app_version väärtusi,
# et kontrollida, kas puuduvad pricing-andmed võivad olla seotud
# konkreetse rakenduse versiooniga

missing_metered["rider_app_version"].value_counts(dropna=False)


rider_app_version
NaN        16
CI.4.17     2
CA.5.26     1
CI.3.22     1
Name: count, dtype: int64

In [40]:
# Kontrollin, kas leidub sõite, millel puuduvad korraga
# kõik peamised hinnastamisega seotud andmed.

pricing_columns = [
    "prediction_price_type",
    "upfront_price",
    "predicted_distance",
    "predicted_duration",
    "change_reason_pricing"
]

missing_pricing_rides = df_clean[
    df_clean[pricing_columns].isna().all(axis=1)
]

print("Puuduvate pricing-andmetega sõite:", len(missing_pricing_rides))

missing_pricing_rides

Puuduvate pricing-andmetega sõite: 20


,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_name,eu_indicator,overpaid_ride_ticket,fraud_score,date,time,day_of_week,day_of_week_nr,month,year
64,217,217,2020-02-04 08:07:10,NaN,NaN,6249,2477,1,driver,finished,...,Xiaomi Redmi 8,1,0,NaN,2020-02-04,08:07:10,Tuesday,2,2,2020
393,3066,3066,2020-03-02 17:45:17,NaN,NaN,5483,917,1,driver,finished,...,HUAWEI EML-L29,1,0,NaN,2020-03-02,17:45:17,Monday,1,3,2020
458,3320,3320,2020-03-06 03:33:52,NaN,NaN,3364,323,0,reseller,finished,...,HUAWEI CLT-L29,1,0,NaN,2020-03-06,03:33:52,Friday,5,3,2020
513,1166,1166,2020-02-13 14:55:14,NaN,NaN,21997,1742,1,driver,finished,...,HUAWEI SLA-L22,1,0,-74.0,2020-02-13,14:55:14,Thursday,4,2,2020
779,1759,1759,2020-02-19 00:55:07,NaN,NaN,10309,849,1,reseller,finished,...,"iPhone8,1",1,0,NaN,2020-02-19,00:55:07,Wednesday,3,2,2020
998,861,861,2020-02-10 06:52:29,NaN,NaN,1424,416,1,driver,finished,...,Samsung SM-G973F,1,0,NaN,2020-02-10,06:52:29,Monday,1,2,2020
1206,1349,1349,2020-02-14 23:08:22,NaN,NaN,11206,1268,1,client,finished,...,Samsung SM-T295,1,0,-1270.0,2020-02-14,23:08:22,Friday,5,2,2020
1287,2012,2012,2020-02-21 13:24:40,NaN,NaN,3717,713,1,driver,finished,...,Samsung SM-A530F,1,0,NaN,2020-02-21,13:24:40,Friday,5,2,2020
1340,1112,1112,2020-02-12 23:32:11,NaN,NaN,4114,528,1,driver,finished,...,Xiaomi Mi A1,1,0,NaN,2020-02-12,23:32:11,Wednesday,3,2,2020
1582,460,460,2020-02-06 19:34:43,NaN,NaN,6718,764,1,client,finished,...,"iPhone8,1",1,0,NaN,2020-02-06,19:34:43,Thursday,4,2,2020


### Puuduva 'upfront_price' analüüs

In [41]:
# Vaatan ainult neid sõite, kus upfront_price puudub,
# ja kontrollin, millised prediction_price_type väärtused neil esinevad

df_clean[
    df_clean["upfront_price"].isna()
]["prediction_price_type"].value_counts(dropna=False)

prediction_price_type
prediction                     1279
upfront_destination_changed     208
upfront                          23
NaN                              20
upfront_waypoint_changed          4
Name: count, dtype: int64

### Olemasoleva 'upfront_price' võrdlus

In [42]:
# Vaatan sõite, kus upfront_price ON olemas,
# ja kontrollin nende prediction_price_type väärtusi

df_clean[
    df_clean["upfront_price"].notna()
]["prediction_price_type"].value_counts(dropna=False)

prediction_price_type
upfront    3409
Name: count, dtype: int64

### Upfront-tüüpi sõidud puuduva upfront_price'iga

In [43]:
# Valin sõidud, kus prediction_price_type on "upfront",
# kuid upfront_price puudub

missing_upfront = df_clean[
    (df_clean["prediction_price_type"] == "upfront") &
    (df_clean["upfront_price"].isna())
]

# Määran veerud, mille väärtusi tahan nende sõitude puhul uurida

columns_to_check = [
    "gps_confidence",
    "dest_change_number",
    "change_reason_pricing",
    "entered_by",
    "b_state",
    "rider_app_version",
    "device_name"
]

# Kuvan iga valitud tunnuse väärtused ja nende esinemissageduse,
# et leida võimalikke ühiseid mustreid

for column in columns_to_check:
    print("\n---", column, "---")
    print(missing_upfront[column].value_counts(dropna=False))


--- gps_confidence ---
gps_confidence
0    23
Name: count, dtype: int64

--- dest_change_number ---
dest_change_number
1    21
5     2
Name: count, dtype: int64

--- change_reason_pricing ---
change_reason_pricing
NaN    23
Name: count, dtype: int64

--- entered_by ---
entered_by
client    21
driver     2
Name: count, dtype: int64

--- b_state ---
b_state
finished    23
Name: count, dtype: int64

--- rider_app_version ---
rider_app_version
CI.4.19    6
CI.4.17    5
CI.4.18    3
CI.4.22    2
CA.5.42    2
CA.5.43    2
CA.5.44    1
CI.4.00    1
CI.3.91    1
Name: count, dtype: int64

--- device_name ---
device_name
HUAWEI VNS-L21        2
Samsung SM-A530F      2
Samsung SM-A520F      2
Samsung SM-J610FN     2
Coolpad E502          2
HUAWEI ATU-L21        2
HUAWEI PRA-LX1        1
Samsung SM-A202F      1
iPhone9,3             1
Samsung SM-G960F      1
Samsung SM-G973F      1
HUAWEI VNS-L31        1
Samsung SM-N975F      1
iPhone8,1             1
HTC One X10           1
HMD Global TA-1053 

## 7. Hindade ja anomaaliate kontroll

### EU ja mitte-EU sõitude hinnatasemete võrdlus

In [44]:
# Võrdlen hindu EU ja mitte-EU sõitude vahel,
# et kontrollida, kas hinnatasemed erinevad oluliselt.
# See aitab vältida eri hinnaskaalade ekslikku võrdlemist.

df_clean.groupby("eu_indicator")["metered_price"].describe()

,count,mean,std,min,25%,50%,75%,max
eu_indicator,,,,,,,,
0,2173.0,18111.117460,19588.005340,2000.0,6312.18,12768.240,20753.20,194483.52
1,2750.0,7.642164,7.058966,2.0,3.99,5.895,8.79,119.43


In [45]:
# Kontrollin, kas väga suured metered_price väärtused
# esinevad ainult mitte-EU sõitudel.

df_clean[
    df_clean["metered_price"] > 1000
]["eu_indicator"].value_counts(dropna=False)

eu_indicator
0    2173
Name: count, dtype: int64

### Ebaloogiliste sõitude kontroll

In [46]:
# Kontrollin eraldi, kui palju sõite sisaldab
# ebaloogilist distantsi või kestust.

print("Distance <= 0:", (df_clean["distance"] <= 0).sum())
print("Duration <= 0:", (df_clean["duration"] <= 0).sum())

Distance <= 0: 64
Duration <= 0: 25


In [47]:
# Lisan tunnuse, mis märgistab ebaloogilise distantsi või kestusega sõidud.
# Neid ridu ei kustutata, sest need võivad olla hilisema analüüsi jaoks olulised.

df_clean["is_invalid_ride"] = (
    (df_clean["distance"] <= 0) |
    (df_clean["duration"] <= 0)
)

# Kontrollin, mitu sõitu märgistati ebaloogiliseks.

df_clean["is_invalid_ride"].value_counts()

is_invalid_ride
False    4879
True       64
Name: count, dtype: int64

### 'fraud_score' jaotuse kontroll

In [48]:
# Uurin fraud_score jaotust.
# Negatiivseid väärtusi ei muuda, sest me ei tea veel,
# milline on selle skoori lubatud skaala.

df_clean["fraud_score"].describe()

count     2184.000000
mean      -674.046703
std       1119.189890
min     -14225.000000
25%       -826.500000
50%       -278.500000
75%        -64.750000
max         49.000000
Name: fraud_score, dtype: float64

In [49]:
# Loendan negatiivsed, null- ja positiivsed fraud_score väärtused.

print("Negative:", (df_clean["fraud_score"] < 0).sum())
print("Zero:", (df_clean["fraud_score"] == 0).sum())
print("Positive:", (df_clean["fraud_score"] > 0).sum())
print("Missing:", df_clean["fraud_score"].isna().sum())

Negative: 2002
Zero: 140
Positive: 42
Missing: 2759


In [50]:
# Loon fraud_score väärtusest kategoorilise abiveeru,
# et fraud_score gruppe oleks hiljem Power BI-s lihtsam
# filtreerida, võrrelda ja visualiseerida.

df_clean["fraud_score_status"] = np.select(
    [
        df_clean["fraud_score"].isna(),
        df_clean["fraud_score"] < 0,
        df_clean["fraud_score"] == 0,
        df_clean["fraud_score"] > 0
    ],
    [
        "Missing",
        "Negative",
        "Zero",
        "Positive"
    ],
    default="Unknown"
)

# Kontrollin loodud kategooriate jaotust.
df_clean["fraud_score_status"].value_counts(dropna=False)

fraud_score_status
Missing     2759
Negative    2002
Zero         140
Positive      42
Name: count, dtype: int64

## 8. Lõplik andmekvaliteedi kontroll

In [51]:
# Teen enne faili salvestamist lõpliku andmekvaliteedi kontrolli.
# Kontrollin andmestiku suurust, andmetüüpe, puuduvaid väärtusi
# ja täielikult identseid ridu.

print("Ridu ja veerge:", df_clean.shape)
print("\nDuplikaatridu:", df_clean.duplicated().sum())

print("\nPuuduvad väärtused:")
print(
    df_clean.isnull()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

df_clean.info()

Ridu ja veerge: (4943, 33)

Duplikaatridu: 0

Puuduvad väärtused:
change_reason_pricing    4645
fraud_score              2759
upfront_price            1534
metered_price              20
predicted_distance         20
prediction_price_type      20
predicted_duration         20
rider_app_version          16
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 4943 entries, 0 to 4942
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   order_id_new           4943 non-null   int64         
 1   order_try_id_new       4943 non-null   int64         
 2   calc_created           4943 non-null   datetime64[us]
 3   metered_price          4923 non-null   float64       
 4   upfront_price          3409 non-null   float64       
 5   distance               4943 non-null   int64         
 6   duration               4943 non-null   int64         
 7   gps_confidence         4943 non-null   int64     

## 9. Puhastatud andmestiku loomine

In [ ]:
# Salvestan puhastatud ja täiendatud andmestiku eraldi CSV-failina.
# df_clean sisaldab:
# - eemaldatud ebavajalikku device_token veergu
# - parandatud calc_created datetime formaati
# - lisatud kuupäevatunnuseid (date, time, day_of_week,
#   day_of_week_nr, month, year)
# - lisatud is_invalid_ride tunnust ebaloogiliste sõitude märgistamiseks
# - lisatud fraud_score_status tunnust Power BI analüüsi lihtsustamiseks
# - kõiki seni säilitatud algandmeid ja anomaaliaid
#
# index=False tähendab, et Pandase reaindeksit CSV-faili ei lisata.

df_clean.to_csv("takso_andmed_clean.csv", index=False)

### Eksporditud faili kontroll

In [ ]:
# Laen loodud clean CSV uuesti sisse,
# et kontrollida faili suurust ja veerge.

df_check = pd.read_csv("takso_andmed_clean.csv")

print("Ridu:", df_check.shape[0])
print("Veerge:", df_check.shape[1])

print("\nVeerud:")
print(df_check.columns.tolist())

Ridu: 4943
Veerge: 33

Veerud:
['order_id_new', 'order_try_id_new', 'calc_created', 'metered_price', 'upfront_price', 'distance', 'duration', 'gps_confidence', 'entered_by', 'b_state', 'dest_change_number', 'prediction_price_type', 'predicted_distance', 'predicted_duration', 'change_reason_pricing', 'ticket_id_new', 'rider_app_version', 'order_state', 'order_try_state', 'driver_app_version', 'driver_device_uid_new', 'device_name', 'eu_indicator', 'overpaid_ride_ticket', 'fraud_score', 'date', 'time', 'day_of_week', 'day_of_week_nr', 'month', 'year', 'is_invalid_ride', 'fraud_score_status']
